If you mean the Python **Selenium** library: Selenium is used to **control and automate a web browser** from Python. For a network engineer, it can be useful when a network appliance or portal exposes important functions only through a **web GUI** rather than SSH/API.

[Selenium official documentation](https://www.selenium.dev/documentation/?utm_source=chatgpt.com)

A simple way to think about it is:

```text
Python
  ↓
Selenium
  ↓
Chrome / Edge / Firefox
  ↓
Web GUI
  ↓
Network device / portal
```

## 1. Install Selenium

```bash
pip install selenium
```

A basic Chrome example:

```python
from selenium import webdriver

driver = webdriver.Chrome()

driver.get("https://www.google.com")

print(driver.title)

driver.quit()
```

Here Selenium opens Chrome, navigates to the website, reads the page title, and closes Chrome.

## 2. How is this useful for a network engineer?

Suppose you have:

```text
Firewall GUI
Router GUI
EVE-NG
Network monitoring portal
Internal network dashboard
IPAM portal
Inventory portal
```

You normally perform:

```text
Open browser
   ↓
Enter URL
   ↓
Login
   ↓
Navigate through menus
   ↓
Check interface/BGP/device status
   ↓
Collect result
```

Selenium can automate much of that browser interaction.

For example:

```text
Python Selenium script
        ↓
Open network portal
        ↓
Authenticate
        ↓
Open device status page
        ↓
Read interface status
        ↓
Capture information
        ↓
Save result
```

## 3. Find elements on a webpage

One of the most important Selenium concepts is locating HTML elements.

Suppose a webpage contains:

```html
<input id="username">
<input id="password">
<button id="login">Login</button>
```

Python can interact with them:

```python
from selenium import webdriver
from selenium.webdriver.common.by import By

driver = webdriver.Chrome()

driver.get("https://example.com")

username = driver.find_element(By.ID, "username")
password = driver.find_element(By.ID, "password")

username.send_keys("myusername")
password.send_keys("mypassword")

driver.find_element(By.ID, "login").click()
```

In real automation, don't hard-code passwords in the script; use environment variables or an approved secrets store.

## 4. Check whether a network portal is accessible

This can be useful for simple portal-health testing.

```python
from selenium import webdriver

driver = webdriver.Chrome()

try:
    driver.get("https://example.com")

    print("Page title:", driver.title)

    if "Network" in driver.title:
        print("Network portal is accessible")
    else:
        print("Unexpected page returned")

finally:
    driver.quit()
```

Notice an important difference:

```text
PING test
    ↓
Is the IP reachable?

TCP test
    ↓
Is port 443 reachable?

HTTP request
    ↓
Is the web server responding?

Selenium
    ↓
Does the actual website/browser workflow work?
```

That makes Selenium useful for **end-to-end GUI validation**.

## 5. Take screenshots

This can be useful during troubleshooting.

```python
from selenium import webdriver

driver = webdriver.Chrome()

driver.get("https://example.com")

driver.save_screenshot("network_portal.png")

driver.quit()
```

For example, an automation could detect that a portal test failed and save:

```text
2026-09-01_10-30-15_failure.png
```

That gives you evidence of what the browser actually displayed.

## 6. Wait for a network dashboard to load

Avoid doing this:

```python
import time

time.sleep(10)
```

A better Selenium approach is an explicit wait:

```python
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

element = WebDriverWait(driver, 10).until(
    EC.presence_of_element_located(
        (By.ID, "device-status")
    )
)

print(element.text)
```

This means:

> Wait up to 10 seconds for `device-status` to appear.

This is much better for network portals because dashboard load times can vary.

## 7. Selenium + network monitoring

Imagine an internal network dashboard showing:

```text
Device        Status
--------------------
R1            UP
R2            UP
R3            DOWN
R4            UP
```

Selenium could open the dashboard and read the table:

```python
rows = driver.find_elements(
    By.CSS_SELECTOR,
    "table tbody tr"
)

for row in rows:
    print(row.text)
```

You could then process the result:

```python
for row in rows:

    data = row.text

    if "DOWN" in data:
        print("WARNING:", data)
```

Now you have basic browser-based network monitoring.

## 8. Selenium + EVE-NG

For a lab environment, you could potentially use Selenium for GUI workflows such as:

```text
Python
 ↓
Selenium
 ↓
Open EVE-NG
 ↓
Navigate to lab
 ↓
Check displayed lab/device state
 ↓
Capture screenshot
 ↓
Record result
```

For normal device configuration, however, Selenium would **not** be my first choice.

Prefer:

```text
Netmiko
NAPALM
REST API
NETCONF
RESTCONF
PyATS
Ansible
```

when the device supports them.

Selenium is more appropriate when:

```text
        Does an API exist?
              |
       +------+------+
       |             |
      YES            NO
       |             |
   Use API       Is there GUI?
                     |
                    YES
                     |
                 Selenium
```

## 9. Selenium + Rich

The `rich` library from your previous question combines nicely with Selenium.

For example:

```python
from rich.console import Console

console = Console()

if portal_working:
    console.print(
        "[bold green]Portal test PASSED[/bold green]"
    )
else:
    console.print(
        "[bold red]Portal test FAILED[/bold red]"
    )
```

You could eventually build:

```text
╔══════════════════════════════════════════════╗
║       NETWORK WEB HEALTH CHECK              ║
╠═══════════════╦══════════════╦══════════════╣
║ Service       ║ Status       ║ Load Time    ║
╠═══════════════╬══════════════╬══════════════╣
║ EVE-NG        ║ ✓ UP         ║ 1.2 sec      ║
║ Monitoring    ║ ✓ UP         ║ 0.8 sec      ║
║ Firewall GUI  ║ ✗ DOWN       ║ Timeout      ║
║ IPAM          ║ ✓ UP         ║ 1.5 sec      ║
╚═══════════════╩══════════════╩══════════════╝
```

## 10. Where Selenium fits in your network-engineering Python toolbox

| Library/tool | Main network-engineering purpose |
|---|---|
| `subprocess` | Ping/system commands |
| `socket` | TCP/UDP testing |
| `requests` | REST APIs / HTTP testing |
| `Netmiko` | SSH to routers/switches |
| `NAPALM` | Multi-vendor automation |
| `pyATS/Genie` | Network testing/parsing |
| `Scapy` | Packet creation/analysis |
| `Rich` | Better terminal output |
| **Selenium** | **Browser/GUI automation** |
| `pandas` | Analyze network data |

So I would learn Selenium mainly for **network web portals, GUI testing, screenshots, dashboards, and browser-based workflows**. For router/switch configuration and operational commands, learn APIs, Netmiko, NETCONF/RESTCONF, and PyATS first.

In [2]:
!pip install selenium

Defaulting to user installation because normal site-packages is not writeable
Looking in indexes: https://packagefeedproxy.microsoft.io/pypi/simple/


In [3]:
from selenium import webdriver

driver = webdriver.Chrome()

driver.get("https://www.google.com")

print(driver.title)

driver.quit()

Google


In [4]:
from selenium import webdriver
from selenium.webdriver.common.by import By

driver = webdriver.Chrome()

driver.get("https://example.com")

username = driver.find_element(By.ID, "username")
password = driver.find_element(By.ID, "password")

username.send_keys("myusername")
password.send_keys("mypassword")

driver.find_element(By.ID, "login").click()

NoSuchElementException: Message: no such element: Unable to locate element: {"method":"css selector","selector":"[id="username"]"}
  (Session info: chrome=152.0.7977.65); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#nosuchelementexception
Stacktrace:
	chromedriver!GetHandleVerifier [0x7ff717ee8a65+5625]
	chromedriver!(No symbol) [0x7ff717e0ce10]
	chromedriver!(No symbol) [0x7ff717c35f8d]
	chromedriver!(No symbol) [0x7ff717c913ea]
	chromedriver!(No symbol) [0x7ff717c916fc]
	chromedriver!(No symbol) [0x7ff717ce22f7]
	chromedriver!(No symbol) [0x7ff717cdeef1]
	chromedriver!(No symbol) [0x7ff717c8384b]
	chromedriver!(No symbol) [0x7ff717c84783]
	chromedriver!GetHandleVerifier [0x7ff71830476b+42132b]
	chromedriver!GetHandleVerifier [0x7ff71832fcf2+44c8b2]
	chromedriver!GetHandleVerifier [0x7ff718323c9e+44085e]
	chromedriver!GetHandleVerifier [0x7ff717fe4e3e+1019fe]
	chromedriver!(No symbol) [0x7ff717e19f6c]
	chromedri

In [5]:
from selenium import webdriver
from selenium.webdriver.common.by import By

driver = webdriver.Chrome()

driver.get("https://github.com")

username = driver.find_element(By.ID, "username")
password = driver.find_element(By.ID, "password")

username.send_keys("myusername")
password.send_keys("mypassword")

driver.find_element(By.ID, "login").click()

NoSuchElementException: Message: no such element: Unable to locate element: {"method":"css selector","selector":"[id="username"]"}
  (Session info: chrome=152.0.7977.65); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#nosuchelementexception
Stacktrace:
	chromedriver!GetHandleVerifier [0x7ff717ee8a65+5625]
	chromedriver!(No symbol) [0x7ff717e0ce10]
	chromedriver!(No symbol) [0x7ff717c35f8d]
	chromedriver!(No symbol) [0x7ff717c913ea]
	chromedriver!(No symbol) [0x7ff717c916fc]
	chromedriver!(No symbol) [0x7ff717ce22f7]
	chromedriver!(No symbol) [0x7ff717cdeef1]
	chromedriver!(No symbol) [0x7ff717c8384b]
	chromedriver!(No symbol) [0x7ff717c84783]
	chromedriver!GetHandleVerifier [0x7ff71830476b+42132b]
	chromedriver!GetHandleVerifier [0x7ff71832fcf2+44c8b2]
	chromedriver!GetHandleVerifier [0x7ff718323c9e+44085e]
	chromedriver!GetHandleVerifier [0x7ff717fe4e3e+1019fe]
	chromedriver!(No symbol) [0x7ff717e19f6c]
	chromedri

In [6]:
from selenium import webdriver

driver = webdriver.Chrome()

try:
    driver.get("https://example.com")

    print("Page title:", driver.title)

    if "Network" in driver.title:
        print("Network portal is accessible")
    else:
        print("Unexpected page returned")

finally:
    driver.quit()

Page title: Example Domain
Unexpected page returned


In [7]:
from selenium import webdriver

driver = webdriver.Chrome()

try:
    driver.get("https://github.com")

    print("Page title:", driver.title)

    if "Network" in driver.title:
        print("Network portal is accessible")
    else:
        print("Unexpected page returned")

finally:
    driver.quit()

Page title: GitHub · Change is constant. GitHub keeps you ahead. · GitHub
Unexpected page returned


In [8]:
from selenium import webdriver

driver = webdriver.Chrome()

try:
    driver.get("https://youtube.com")

    print("Page title:", driver.title)

    if "Network" in driver.title:
        print("Network portal is accessible")
    else:
        print("Unexpected page returned")

finally:
    driver.quit()

Page title: YouTube
Unexpected page returned


. Take screenshots
This can be useful during troubleshooting.

In [12]:
from selenium import webdriver

driver = webdriver.Chrome()

driver.get("https://example.com")

driver.save_screenshot("example.png")

driver.quit()

In [13]:
from selenium import webdriver

driver = webdriver.Chrome()

driver.get("https://youtube.com")

driver.save_screenshot("youtube.png")

driver.quit()

In [14]:
from selenium import webdriver

driver = webdriver.Chrome()

driver.get("https://github.com")

driver.save_screenshot("github.png")

driver.quit()

In [15]:
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

element = WebDriverWait(driver, 10).until(
    EC.presence_of_element_located(
        (By.ID, "device-status")
    )
)

print(element.text)

MaxRetryError: HTTPConnectionPool(host='localhost', port=64992): Max retries exceeded with url: /session/606596d5fd103bead4c08c0c1cee1b61/element (Caused by NewConnectionError("HTTPConnection(host='localhost', port=64992): Failed to establish a new connection: [WinError 10061] No connection could be made because the target machine actively refused it"))

In [16]:
from rich.console import Console

console = Console()

if portal_working:
    console.print(
        "[bold green]Portal test PASSED[/bold green]"
    )
else:
    console.print(
        "[bold red]Portal test FAILED[/bold red]"
    )

NameError: name 'portal_working' is not defined